# 01 — 核心概念与状态管理

**来源:** [菜鸟教程 — LangGraph 入门教程](https://www.runoob.com/ai-agent/langgraph-quick-start.html)

LangGraph 是由 LangChain 团队开发的低层级 Agent 编排框架，专为构建有状态、长时运行的 AI 工作流而设计。

## 1. 核心概念

### 1.1 Graph（图）
Graph 是整个工作流的蓝图，由节点（Nodes）和边（Edges）组成。

```
StateGraph
├── Nodes（节点）
│   ├── node_a
│   ├── node_b
│   └── node_c
└── Edges（边）
    ├── START -> node_a
    ├── node_a -> node_b（条件边）
    ├── node_a -> node_c（条件边）
    └── node_b -> END
```

### 1.2 State（状态）
State 是贯穿整个图的共享数据结构。每个节点可以读取和更新 State。
使用 `TypedDict` 定义，支持通过 `Annotated` 指定 reducer（如 `add_messages`）确保消息自动追加而非覆盖。

### 1.3 Nodes（节点）
普通 Python 函数，接收当前 State，返回部分字段更新。

### 1.4 Edges（边）
- **普通边：** 固定路径
- **条件边：** 根据 State 动态路由
- **起始边：** `START -> 第一个节点`
- **结束边：** `某节点 -> END`

### 与传统 LLM Chain 对比

| 特性 | 传统 LLM Chain | LangGraph |
|------|----------------|-----------|
| 工作流结构 | 线性，单向执行 | 图结构，支持循环 |
| 状态管理 | 需手动管理 | 内置状态持久化 |
| 条件路由 | 实现复杂 | 原生支持 |
| 人机协作 | 需要额外开发 | 内置 interrupt |
| 多 Agent 协调 | 实现困难 | 一流支持 |
| 调试工具 | 有限 | LangGraph Studio |

**适用场景：** 对话机器人、自主 Agent、多 Agent 系统、审批工作流、研究助手。

## 2. State 状态管理

### 2.1 使用 TypedDict 定义（含 reducer）

`add_messages` 是最常用的 reducer，确保新消息追加到列表而非覆盖。
自定义 reducer 可用于计数器累加等场景。

In [1]:
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages


class MyState(TypedDict):
    """使用 TypedDict 定义的 State"""
    messages: Annotated[list, add_messages]  # 消息列表（自动追加）
    user_name: str                            # 用户名称
    step_count: int                           # 步骤计数

print("MyState 定义完成")


MyState 定义完成


重要：`Annotated[list, add_messages]` 表示该字段使用 `add_messages` 作为 reducer——新消息会追加到列表而不是覆盖。这是 LangGraph 状态管理的核心机制。

In [2]:
from typing import TypedDict, Annotated, Optional
from langgraph.graph.message import add_messages


class AgentState(TypedDict):
    """使用 TypedDict 定义的状态（进阶）"""
    messages: Annotated[list, add_messages]  # 消息列表，自动追加而非覆盖
    user_id: str  # 普通字段，直接覆盖
    session_id: str
    error: Optional[str]  # 可选字段
    retry_count: Annotated[int, lambda x, y: x + y]  # 自定义 reducer：累加器


print("AgentState 定义完成")


AgentState 定义完成


### 2.2 使用 Pydantic（推荐生产环境）

Pydantic 提供类型验证、默认值和配置选项，适合生产环境。

In [ ]:
from pydantic import BaseModel, Field
from typing import Annotated
from langgraph.graph.message import add_messages


class ProductionState(BaseModel):
    """使用 Pydantic 定义的 State，推荐生产环境使用"""
    messages: Annotated[list, add_messages] = Field(default_factory=list)
    user_id: str = ""
    confidence_score: float = 0.0

    class Config:
        arbitrary_types_allowed = True


print("ProductionState 定义完成")


### 2.3 MessagesState（内置快捷状态）

LangGraph 提供了内置的 `MessagesState`，专为对话场景设计：

In [4]:
from langgraph.graph import MessagesState, StateGraph

# MessagesState 等价于:
# class MessagesState(TypedDict):
#     messages: Annotated[list[AnyMessage], add_messages]

# 直接使用，无需自定义
builder = StateGraph(MessagesState)
print("MessagesState 可直接使用")


MessagesState 可直接使用
